# Setup

In [1]:
!pip install -q catboost
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# Baseline Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Proposed Ensemble Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.9 MB/s eta 0:00:00


In [2]:
def print_evaluation_metrics(model_name, y_true, y_pred, y_prob):
    print(f"=== Performance: {model_name} ===")
    print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision : {precision_score(y_true, y_pred):.4f}")
    print(f"Recall    : {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score  : {f1_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC   : {roc_auc_score(y_true, y_prob):.4f}\n")

In [3]:
current_dir = os.getcwd()
print(f"Started in: {current_dir}")

if current_dir.endswith("notebook"):
    os.chdir("..")

print(f"Now running in ROOT: {os.getcwd()}")

Started in: /content
Now running in ROOT: /content


In [4]:
# read data for baseline models (distance based)
hist_processed_path = "ProcessedHistoric.csv"
df_hist_processed = pd.read_csv(hist_processed_path)
df_hist_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236952 entries, 0 to 236951
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Timestamp         236952 non-null  object 
 1   Temperature       236952 non-null  float64
 2   Humidity          236952 non-null  float64
 3   Cloud_Cover       236952 non-null  float64
 4   Pressure          236952 non-null  float64
 5   Wind_Speed        236952 non-null  float64
 6   Location          236952 non-null  object 
 7   Latitude          236952 non-null  float64
 8   Longitude         236952 non-null  float64
 9   Rain              236952 non-null  int64  
 10  Hour              236952 non-null  int64  
 11  Month             236952 non-null  int64  
 12  Location_Encoded  236952 non-null  int64  
 13  hour_sin          236952 non-null  float64
 14  hour_cos          236952 non-null  float64
 15  month_sin         236952 non-null  float64
 16  month_cos         23

In [ ]:
# read data for proposed ensemble models (decision tree based)
hist_path = "historic.csv"
df_hist = pd.read_csv(hist_path)
df_hist.info()

<class 'pandas.DataFrame'>
RangeIndex: 236952 entries, 0 to 236951
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Timestamp    236952 non-null  str    
 1   Temperature  236952 non-null  float64
 2   Humidity     236952 non-null  int64  
 3   Cloud_Cover  236952 non-null  int64  
 4   Pressure     236952 non-null  float64
 5   Wind_Speed   236952 non-null  float64
 6   Location     236952 non-null  str    
 7   Latitude     236952 non-null  float64
 8   Longitude    236952 non-null  float64
 9   Rain         236952 non-null  int64  
dtypes: float64(5), int64(3), str(2)
memory usage: 23.8 MB


In [5]:
# dropping unused features (including the target feature 'Rain')
processed_columns_to_drop = [
    "Rain",
    "Hour",
    "Month",
    "Latitude",
    "Longitude",
    "Timestamp",
    "Location",
    "Location_Encoded"
]
x_processed = df_hist_processed.drop(columns=processed_columns_to_drop, errors="ignore")
x_processed.info()
y_processed = df_hist_processed["Rain"]

# Split into training and internal validation sets
X_processed_train, X_processed_val, y_processed_train, y_processed_val = (
    train_test_split(
        x_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed
    )
)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236952 entries, 0 to 236951
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Temperature  236952 non-null  float64
 1   Humidity     236952 non-null  float64
 2   Cloud_Cover  236952 non-null  float64
 3   Pressure     236952 non-null  float64
 4   Wind_Speed   236952 non-null  float64
 5   hour_sin     236952 non-null  float64
 6   hour_cos     236952 non-null  float64
 7   month_sin    236952 non-null  float64
 8   month_cos    236952 non-null  float64
dtypes: float64(9)
memory usage: 16.3 MB


In [ ]:
columns_to_drop = ["Rain", "Timestamp", "Location", "Latitude", "Longitude"]
X = df_hist.drop(columns=columns_to_drop, errors="ignore")
X.info()
y = df_hist["Rain"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.sample(n=20000, random_state=42)
y_train = y_train.loc[X_train.index]

<class 'pandas.DataFrame'>
RangeIndex: 236952 entries, 0 to 236951
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Temperature  236952 non-null  float64
 1   Humidity     236952 non-null  int64  
 2   Cloud_Cover  236952 non-null  int64  
 3   Pressure     236952 non-null  float64
 4   Wind_Speed   236952 non-null  float64
dtypes: float64(3), int64(2)
memory usage: 9.0 MB


# Train Baseline Models

In [ ]:
print("--- TRAINING LOGISTIC REGRESSION ---")
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_processed_train, y_processed_train)
logistic_pred = logistic_model.predict(X_processed_val)
logistic_prob = logistic_model.predict_proba(X_processed_val)[:, 1]
print_evaluation_metrics(
    "Logistic Regression", y_processed_val, logistic_pred, logistic_prob
)

--- TRAINING LOGISTIC REGRESSION ---
=== Performance: Logistic Regression ===
Accuracy  : 0.6947
Precision : 0.5517
Recall    : 0.2648
F1-Score  : 0.3578
ROC-AUC   : 0.7158



In [ ]:
print("--- TRAINING SUPPORT VECTOR MACHINE ---")
svm_model = SVC(probability=True, random_state=42, class_weight="balanced")
svm_model.fit(X_processed_train, y_processed_train)
svm_pred = svm_model.predict(X_processed_val)
svm_prob = svm_model.predict_proba(X_processed_val)[:, 1]
print_evaluation_metrics("Support Vector Machine", y_processed_val, svm_pred, svm_prob)

--- TRAINING SUPPORT VECTOR MACHINE ---
=== Performance: Support Vector Machine ===
Accuracy  : 0.4876
Precision : 0.3732
Recall    : 0.8759
F1-Score  : 0.5234
ROC-AUC   : 0.6732



In [ ]:
print("--- TRAINING RANDOM FOREST ---")
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_processed_train, y_processed_train)
rf_pred = rf_model.predict(X_processed_val)
rf_prob = rf_model.predict_proba(X_processed_val)[:, 1]
print_evaluation_metrics("Random Forest", y_processed_val, rf_pred, rf_prob)

--- TRAINING RANDOM FOREST ---
=== Performance: Random Forest ===
Accuracy  : 0.7376
Precision : 0.6260
Recall    : 0.4551
F1-Score  : 0.5271
ROC-AUC   : 0.7807



In [ ]:
print("--- TRAINING DECISION TREE ---")
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_processed_train, y_processed_train)
dt_pred = dt_model.predict(X_processed_val)
dt_prob = dt_model.predict_proba(X_processed_val)[:, 1]
print_evaluation_metrics("Decision Tree", y_processed_val, dt_pred, dt_prob)

--- TRAINING DECISION TREE ---
=== Performance: Decision Tree ===
Accuracy  : 0.6797
Precision : 0.5015
Recall    : 0.5095
F1-Score  : 0.5055
ROC-AUC   : 0.6350



# Train proposed ensemble models (XGBoost, LightGBM, CatBoost)

In [ ]:
print("\n--- TRAINING PROPOSED ENSEMBLE (XGBoost, LightGBM, CatBoost) ---")
xgb = XGBClassifier(random_state=42, eval_metric="logloss")
lgb = LGBMClassifier(random_state=42, verbose=-1)
cat = CatBoostClassifier(random_state=42, verbose=0)

xgb.fit(X_train, y_train)
lgb.fit(X_train, y_train)
cat.fit(X_train, y_train)

xgb_pred = xgb.predict(X_val)
lgb_pred = lgb.predict(X_val)
cat_pred = cat.predict(X_val)

prob_xgb = xgb.predict_proba(X_val)[:, 1]
prob_lgb = lgb.predict_proba(X_val)[:, 1]
prob_cat = cat.predict_proba(X_val)[:, 1]

print_evaluation_metrics("XGBoost", y_val, xgb_pred, prob_xgb)
print_evaluation_metrics("LightGBM", y_val, lgb_pred, prob_lgb)
print_evaluation_metrics("CatBoost", y_val, cat_pred, prob_cat)


--- TRAINING PROPOSED ENSEMBLE (XGBoost, LightGBM, CatBoost) ---
=== Performance: XGBoost ===
Accuracy  : 0.7352
Precision : 0.6217
Recall    : 0.4486
F1-Score  : 0.5211
ROC-AUC   : 0.7795

=== Performance: LightGBM ===
Accuracy  : 0.7388
Precision : 0.6393
Recall    : 0.4289
F1-Score  : 0.5134
ROC-AUC   : 0.7843

=== Performance: CatBoost ===
Accuracy  : 0.7404
Precision : 0.6422
Recall    : 0.4333
F1-Score  : 0.5174
ROC-AUC   : 0.7882



In [ ]:
# all equal weights
w1_xgb, w1_lgb, w1_cat = 0.34, 0.33, 0.33

ensemble1_prob_val = (w1_xgb * prob_xgb) + (w1_lgb * prob_lgb) + (w1_cat * prob_cat)
ensemble1_pred_val = (ensemble1_prob_val >= 0.5).astype(int)

print_evaluation_metrics(
    "Weighted Ensemble (Proposed)", y_val, ensemble1_pred_val, ensemble1_prob_val
)

=== Performance: Weighted Ensemble (Proposed) ===
Accuracy  : 0.7408
Precision : 0.6419
Recall    : 0.4371
F1-Score  : 0.5201
ROC-AUC   : 0.7876



In [ ]:
# different weight 2
w2_xgb, w2_lgb, w2_cat = 0.40, 0.15, 0.45

ensemble2_prob_val = (w2_xgb * prob_xgb) + (w2_lgb * prob_lgb) + (w2_cat * prob_cat)
ensemble2_pred_val = (ensemble2_prob_val >= 0.5).astype(int)

print_evaluation_metrics(
    "Weighted Ensemble (Proposed)", y_val, ensemble2_pred_val, ensemble2_prob_val
)

=== Performance: Weighted Ensemble (Proposed) ===
Accuracy  : 0.7405
Precision : 0.6400
Recall    : 0.4389
F1-Score  : 0.5207
ROC-AUC   : 0.7876



In [ ]:
# different weight 3
w3_xgb, w3_lgb, w3_cat = 0.50, 0.15, 0.35

ensemble3_prob_val = (w3_xgb * prob_xgb) + (w3_lgb * prob_lgb) + (w3_cat * prob_cat)
ensemble3_pred_val = (ensemble3_prob_val >= 0.5).astype(int)

print_evaluation_metrics(
    "Weighted Ensemble (Proposed)", y_val, ensemble3_pred_val, ensemble3_prob_val
)

=== Performance: Weighted Ensemble (Proposed) ===
Accuracy  : 0.7402
Precision : 0.6388
Recall    : 0.4399
F1-Score  : 0.5210
ROC-AUC   : 0.7867



In [ ]:
# different weight 4
w4_xgb, w4_lgb, w4_cat = 0.25, 0.20, 0.55

ensemble4_prob_val = (w4_xgb * prob_xgb) + (w4_lgb * prob_lgb) + (w4_cat * prob_cat)
ensemble4_pred_val = (ensemble4_prob_val >= 0.5).astype(int)

print_evaluation_metrics(
    "Weighted Ensemble (Proposed)", y_val, ensemble4_pred_val, ensemble4_prob_val
)

=== Performance: Weighted Ensemble (Proposed) ===
Accuracy  : 0.7410
Precision : 0.6430
Recall    : 0.4358
F1-Score  : 0.5195
ROC-AUC   : 0.7885



We will use weight 3, as it has the highest recall and f1-score meaning it has the lowest false negative rate which in our case important to minimize

In [10]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 1. Initialize base models
xgb_base = XGBClassifier(random_state=42, eval_metric="logloss")
lgb_base = LGBMClassifier(random_state=42, verbose=-1)
cat_base = CatBoostClassifier(random_state=42, verbose=0)

# 2. Define stronger parameter grids
xgb_param_dist = {
    'n_estimators': [200, 300, 500],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

lgb_param_dist = {
    'n_estimators': [200, 300, 500],
    'num_leaves': [31, 50, 70, 100],
    'max_depth': [-1, 5, 10, 15],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

cat_param_dist = {
    'iterations': [200, 300, 500],
    'depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5, 7]
}

# 3. Tune models targeting F1-score with preprocessed data
print("Tuning XGBoost...")
xgb_search = RandomizedSearchCV(xgb_base, xgb_param_dist, n_iter=30, scoring='f1', cv=3, random_state=42, n_jobs=-1)
xgb_search.fit(X_processed_train, y_processed_train)
best_xgb = xgb_search.best_estimator_
print(f"Best XGB Params: {xgb_search.best_params_}")

print("Tuning LightGBM...")
lgb_search = RandomizedSearchCV(lgb_base, lgb_param_dist, n_iter=30, scoring='f1', cv=3, random_state=42, n_jobs=-1)
lgb_search.fit(X_processed_train, y_processed_train)
best_lgb = lgb_search.best_estimator_
print(f"Best LGBM Params: {lgb_search.best_params_}")

print("Tuning CatBoost...")
cat_search = RandomizedSearchCV(cat_base, cat_param_dist, n_iter=30, scoring='f1', cv=3, random_state=42, n_jobs=-1)
cat_search.fit(X_processed_train, y_processed_train)
best_cat = cat_search.best_estimator_
print(f"Best CatBoost Params: {cat_search.best_params_}")

# 4. Get probabilities for optimization using validation set
prob_xgb = best_xgb.predict_proba(X_processed_val)[:, 1]
prob_lgb = best_lgb.predict_proba(X_processed_val)[:, 1]
prob_cat = best_cat.predict_proba(X_processed_val)[:, 1]

# 5. Optimize ensemble weights to maximize F1-Score
print("Optimizing Weights for F1 Score...")
best_f1 = 0
best_weights = (0, 0, 0)

for w_xgb in np.linspace(0, 1, 101):
    for w_lgb in np.linspace(0, 1 - w_xgb, 101):
        w_cat = 1.0 - w_xgb - w_lgb
        if w_cat < -0.001 or w_cat > 1.001:
            continue

        ensemble_prob = (w_xgb * prob_xgb + w_lgb * prob_lgb + w_cat * prob_cat)
        ensemble_pred = (ensemble_prob >= 0.5).astype(int)

        current_f1 = f1_score(y_processed_val, ensemble_pred)
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_weights = (w_xgb, w_lgb, w_cat)

print(f"Best Weights : XGB: {best_weights[0]:.4f}, LGBM: {best_weights[1]:.4f}, CatBoost: {best_weights[2]:.4f}")

# 6. Evaluate final proposed ensemble
ensemble_prob_val = (best_weights[0] * prob_xgb + best_weights[1] * prob_lgb + best_weights[2] * prob_cat)
ensemble_pred_val = (ensemble_prob_val >= 0.5).astype(int)

print_evaluation_metrics("F1-Tuned Weighted Ensemble (Proposed)", y_processed_val, ensemble_pred_val, ensemble_prob_val)

Tuning XGBoost...
Best XGB Params: {'subsample': 0.8, 'n_estimators': 500, 'max_depth': 9, 'learning_rate': 0.2, 'colsample_bytree': 0.8}
Tuning LightGBM...
Best LGBM Params: {'subsample': 0.8, 'num_leaves': 31, 'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.2, 'colsample_bytree': 0.6}
Tuning CatBoost...
Best CatBoost Params: {'learning_rate': 0.2, 'l2_leaf_reg': 5, 'iterations': 300, 'depth': 10}
Optimizing Weights for F1 Score...
Best Weights : XGB: 1.0000, LGBM: 0.0000, CatBoost: 0.0000
=== Performance: F1-Tuned Weighted Ensemble (Proposed) ===
Accuracy  : 0.8811
Precision : 0.8422
Recall    : 0.7750
F1-Score  : 0.8072
ROC-AUC   : 0.9364



In [7]:
!pip install -q onnxmltools skl2onnx onnxruntime
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, fbeta_score, make_scorer
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import onnxmltools
from onnxmltools.convert import convert_xgboost, convert_lightgbm
from skl2onnx import convert_sklearn
from onnxmltools.convert.common.data_types import FloatTensorType as OnnxFloat
from skl2onnx.common.data_types import FloatTensorType as SklFloat
import onnxruntime as rt
import warnings

warnings.filterwarnings('ignore')

def print_evaluation_metrics(model_name, y_true, y_pred, y_prob):
    print(f"=== Performance: {model_name} ===")
    print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision : {precision_score(y_true, y_pred):.4f}")
    print(f"Recall    : {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score  : {f1_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC   : {roc_auc_score(y_true, y_prob):.4f}\n")

# 1. Load Data and Convert to NumPy (Fixes ONNX string bug)
df = pd.read_csv("ProcessedHistoric.csv")
columns_to_drop = ["Rain", "Timestamp", "Location", "Latitude", "Longitude"]

# Extract values as numpy arrays to strip feature names
X = df.drop(columns=columns_to_drop, errors="ignore").values
y = df["Rain"].values

X_processed_train, X_processed_val, y_processed_train, y_processed_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Initialize Base Models
xgb_base = XGBClassifier(random_state=42, eval_metric="logloss", n_jobs=-1)
lgb_base = LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1)
cat_base = CatBoostClassifier(random_state=42, verbose=0, thread_count=-1)

xgb_param_dist = {'n_estimators': [200, 300, 500], 'max_depth': [3, 5, 7, 9], 'learning_rate': [0.01, 0.05, 0.1, 0.2], 'subsample': [0.6, 0.8, 1.0], 'colsample_bytree': [0.6, 0.8, 1.0]}
lgb_param_dist = {'n_estimators': [200, 300, 500], 'num_leaves': [31, 50, 70, 100], 'max_depth': [-1, 5, 10, 15], 'learning_rate': [0.01, 0.05, 0.1, 0.2], 'subsample': [0.6, 0.8, 1.0], 'colsample_bytree': [0.6, 0.8, 1.0]}
cat_param_dist = {'iterations': [200, 300, 500], 'depth': [4, 6, 8, 10], 'learning_rate': [0.01, 0.05, 0.1, 0.2], 'l2_leaf_reg': [1, 3, 5, 7]}

# 3. Tune Base Models for F1-Score
print("Tuning Base Models...")
best_xgb = RandomizedSearchCV(xgb_base, xgb_param_dist, n_iter=30, scoring='f1', cv=3, random_state=42, n_jobs=-1).fit(X_processed_train, y_processed_train).best_estimator_
best_lgb = RandomizedSearchCV(lgb_base, lgb_param_dist, n_iter=30, scoring='f1', cv=3, random_state=42, n_jobs=-1).fit(X_processed_train, y_processed_train).best_estimator_
best_cat = RandomizedSearchCV(cat_base, cat_param_dist, n_iter=30, scoring='f1', cv=3, random_state=42, n_jobs=-1).fit(X_processed_train, y_processed_train).best_estimator_

# 4. Build and Train Stacking Classifier
print("Training Stacking Ensemble...")
estimators = [('xgb', best_xgb), ('lgb', best_lgb), ('cat', best_cat)]
stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(class_weight='balanced'),
    cv=3,
    n_jobs=-1
)
stacking_model.fit(X_processed_train, y_processed_train)

# 5. Optimize Threshold to maximize F1-Score
print("Optimizing Threshold for F1 Score...")
stacking_prob = stacking_model.predict_proba(X_processed_val)[:, 1]
best_f1 = 0
best_threshold = 0.5

for threshold in np.linspace(0.3, 0.7, 41):
    stacking_pred = (stacking_prob >= threshold).astype(int)
    current_f1 = f1_score(y_processed_val, stacking_pred)

    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold

print(f"Best Stacking Threshold: {best_threshold:.3f}")
final_stacking_pred = (stacking_prob >= best_threshold).astype(int)
print_evaluation_metrics("Tuned Stacking Ensemble", y_processed_val, final_stacking_pred, stacking_prob)

# 6. Export Decoupled ONNX Models
print("Exporting decoupled models to ONNX...")
onnx_input_shape = [('input', OnnxFloat([None, X_processed_train.shape[1]]))]

xgb_onnx = convert_xgboost(best_xgb, initial_types=onnx_input_shape)
onnxmltools.utils.save_model(xgb_onnx, 'xgb_model.onnx')

lgb_onnx = convert_lightgbm(best_lgb, initial_types=onnx_input_shape)
onnxmltools.utils.save_model(lgb_onnx, 'lgb_model.onnx')

best_cat.save_model("cat_model.onnx", format="onnx")

meta_input_shape = [('meta_input', SklFloat([None, 3]))]
meta_onnx = convert_sklearn(stacking_model.final_estimator_, initial_types=meta_input_shape)

with open("meta_model.onnx", "wb") as f:
    f.write(meta_onnx.SerializeToString())

print("All models exported successfully.")

# 7. Test ONNX Inference Loading
print("Testing ONNX Inference...")
def get_class_1_prob(onnx_output):
    probs = onnx_output[1]
    if isinstance(probs, list) and isinstance(probs[0], dict):
        return np.array([p[1] for p in probs])
    return probs[:, 1]

sess_xgb = rt.InferenceSession("xgb_model.onnx")
sess_lgb = rt.InferenceSession("lgb_model.onnx")
sess_cat = rt.InferenceSession("cat_model.onnx")
sess_meta = rt.InferenceSession("meta_model.onnx")

# Data is already numpy, just cast type
X_infer = X_processed_val.astype(np.float32)

prob_xgb_onnx = get_class_1_prob(sess_xgb.run(None, {sess_xgb.get_inputs()[0].name: X_infer}))
prob_lgb_onnx = get_class_1_prob(sess_lgb.run(None, {sess_lgb.get_inputs()[0].name: X_infer}))
prob_cat_onnx = get_class_1_prob(sess_cat.run(None, {sess_cat.get_inputs()[0].name: X_infer}))

stacked_features = np.column_stack((prob_xgb_onnx, prob_lgb_onnx, prob_cat_onnx)).astype(np.float32)
final_prob_onnx = get_class_1_prob(sess_meta.run(None, {sess_meta.get_inputs()[0].name: stacked_features}))

final_pred_onnx = (final_prob_onnx >= best_threshold).astype(int)
print_evaluation_metrics("Decoupled ONNX Stacking Ensemble", y_processed_val, final_pred_onnx, final_prob_onnx)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.0/304.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 13.3 MB/s eta 0:00:00
Tuning Base Models...
Training Stacking Ensemble...
Optimizing Threshold for F1 Score...
Best Stacking Threshold: 0.520
=== Performance: Tuned Stacking Ensemble ===
Accuracy  : 0.8682
Precision : 0.7780
Recall    : 0.8253
F1-Score  : 0.8009
ROC-AUC   : 0.9341

Exporting decoupled models to ONNX...
All models exported successfully.
Testing ONNX Inference...
=== Performance: Decoupled ONNX Stacking Ensemble ===
Accuracy  : 0.8684
Precision : 0.7786
Recall    : 0.8249
F1-Score  : 0.8011
ROC-AUC   : 0.9341

